# RSNA Knee — Phase 0 offline inference

**Internet OFF.** Loads the Phase-0 weights + the exact preprocess config from an attached
Kaggle Dataset, runs the *same* `preprocess_study()` used in training, and writes
`submission.csv` with all 12 columns in the required order.

Paths are read from environment variables so the identical notebook runs on Kaggle and on
the synthetic fixture under `pytest` — there is no test-only code path.

| variable | default (Kaggle) |
|---|---|
| `KNEE_CODE_DIR` | `/kaggle/input/knee-phase0/code` |
| `KNEE_WEIGHTS_DIR` | `/kaggle/input/knee-phase0/artifacts` |
| `KNEE_DATA_DIR` | `/kaggle/input/rsna-2026-knee-mri` |
| `KNEE_SPLIT` | `test` |
| `KNEE_OUT` | `submission.csv` |

In [ ]:
import os, sys, time, glob
from pathlib import Path

T_START = time.time()

CODE_DIR = Path(os.environ.get("KNEE_CODE_DIR", "/kaggle/input/knee-phase0/code"))
WEIGHTS_DIR = Path(os.environ.get("KNEE_WEIGHTS_DIR", "/kaggle/input/knee-phase0/artifacts"))
DATA_DIR = Path(os.environ.get("KNEE_DATA_DIR", "/kaggle/input/rsna-2026-knee-mri"))
SPLIT = os.environ.get("KNEE_SPLIT", "test")
OUT_PATH = Path(os.environ.get("KNEE_OUT", "submission.csv"))
BATCH_SIZE = int(os.environ.get("KNEE_BATCH_SIZE", "4"))
NUM_WORKERS = int(os.environ.get("KNEE_NUM_WORKERS", "2"))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR, "| weights:", WEIGHTS_DIR, "| data:", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from config import ID_COLUMN, RunConfig, target_columns
from data import StudyDataset, load_tables
from model import build_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEVICE)

In [ ]:
# --- the exact training-time preprocess config travels with the weights ---
run_cfg = RunConfig.load(WEIGHTS_DIR / "run_config.json")
PRE = run_cfg.preprocess
print("preprocess:", PRE.to_dict())

# Submission header: the competition's sample_submission.csv is authoritative.
SAMPLE = DATA_DIR / "sample_submission.csv"
COLUMNS = target_columns(SAMPLE)
assert len(COLUMNS) == 12, COLUMNS
print("columns:", COLUMNS)

In [ ]:
tables = load_tables(DATA_DIR, split=SPLIT)

# Every study in sample_submission must appear in the output, even if its
# images fail to decode.
if SAMPLE.exists():
    required_uids = pd.read_csv(SAMPLE, dtype={ID_COLUMN: str})[ID_COLUMN].astype(str).tolist()
else:
    required_uids = tables.study_uids
print(f"{len(required_uids)} test studies | {len(tables.series)} series rows")

In [ ]:
model = build_model(backbone=run_cfg.backbone, pretrained=False)  # offline: no download
state = torch.load(WEIGHTS_DIR / "model.pt", map_location="cpu")
model.load_state_dict(state)
model.eval().to(DEVICE)
print("weights loaded")

In [ ]:
t_infer = time.time()
known = [u for u in required_uids if u in set(tables.study_uids)]
ds = StudyDataset(known, tables.series, PRE)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

probs, uids = [], []
with torch.no_grad():
    for batch in loader:
        logits = model(batch["x"].to(DEVICE))
        probs.append(torch.sigmoid(logits).float().cpu().numpy())
        uids.extend(batch["study_uid"])

pred = (
    pd.DataFrame(np.concatenate(probs, axis=0), index=uids, columns=COLUMNS)
    if probs
    else pd.DataFrame(columns=COLUMNS)
)
print(f"inference on {len(pred)} studies in {time.time() - t_infer:.1f}s")

In [ ]:
# Missing / undecodable studies fall back to 0.5 rather than dropping a row —
# a missing StudyInstanceUID invalidates the whole submission.
sub = pd.DataFrame({ID_COLUMN: [str(u) for u in required_uids]})
pred = pred.reindex(sub[ID_COLUMN]).astype(float)
for col in COLUMNS:
    sub[col] = pred[col].fillna(0.5).to_numpy()

assert list(sub.columns) == [ID_COLUMN, *COLUMNS], list(sub.columns)
assert len(sub) == len(required_uids) and sub[ID_COLUMN].is_unique
assert sub[COLUMNS].to_numpy().min() >= 0.0 and sub[COLUMNS].to_numpy().max() <= 1.0
assert not sub[COLUMNS].isna().to_numpy().any()

sub.to_csv(OUT_PATH, index=False)
print(sub.head())
print(f"wrote {OUT_PATH} — {len(sub)} rows x {len(COLUMNS)} probability columns")
print(f"TOTAL WALL CLOCK: {time.time() - T_START:.1f}s (budget 9h)")